In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os 
print(os.listdir('/content/drive/MyDrive/data/names'))

In [ ]:
import torch
import string 
import unicodedata

In [ ]:

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda") 
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")


In [ ]:
allowed_chars = string.ascii_letters + " .,;'" + "_"
n_lets = len(allowed_chars) 

def unicodeToAscii(s): 
    return ''.join(
        c for c in unicodedata.normalize('NFD', s) 
        if unicodedata.category(c) != 'Mn'
        and c if allowed_chars
    )

###
print(f"Converting 'Slusarski' to {unicodeToAscii('Slusarski')}")
###

In [ ]:
def letter_to_indx(letter): 
    if letter not in allowed_chars:
        return allowed_chars.find("_") 
    else: 
        return allowed_chars.find(letter) 

def lineToTensor(line): 
    tensor = torch.zeros(len(line), 1, n_lets)
    for li, letter in enumerate(line): 
        tensor[li][0][letter_to_indx(letter)] = 1
    return tensor 


print(f"The letter 'a' becomes{lineToTensor('a')}")
print(f"The name 'Ahn' becomes {lineToTensor('Ahn')}")


In [ ]:

import glob 
import os 
import time 
import torch 
from torch.utils.data import Dataset 
from io import open 

class NamesDataset(Dataset):

    def __init__(self, data_dir): 
        self.data_dir = data_dir
        self.load_time = time.localtime
        labels_set = set() 

        self.data = []
        self.data_tensors = [] 
        self.labels = [] 
        self.labels_tensors = [] 

        text_files = glob.glob(os.path.join(data_dir, '*.txt')) 
        for filename in text_files: 
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label) 
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines: 
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label) 

        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)): 
            tmp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long) 
            self.labels_tensors.append(tmp_tensor) 


    def __len__(self): 
        return len(self.data) 

    def __getitem__(self, idx):
        data_item = self.data[idx] 
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx] 
        label_tensor = self.labels_tensors[idx] 

        return label_tensor, data_tensor, data_label, data_item

    

In [ ]:
print(os.path.abspath("rneuralnets/data/names"))
print(glob.glob(os.path.join("/content/drive/MyDrive/data/names", "*.txt")))

In [ ]:
alldata = NamesDataset("/content/drive/MyDrive/data/names")
print(f"loaded {len(alldata)} items of data")
print(f"example = {alldata[0]}")


In [ ]:
# print(len(alldata.data), len(alldata.labels), len(alldata.data_tensors))
# print(len(alldata))

In [ ]:
train_set, test_set = torch.utils.data.random_split(alldata, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))
print(f"train examples = {len(train_set)}, validation examples = {len(test_set)}")


In [ ]:
import torch.nn as nn 
import torch.nn.functional as F

class CharRNN(nn.Module):
    def __init__(self, num_layers, inp_size, hid_size, hid_size_lstm, outp_size, hid_size_gru): 
        super(CharRNN, self).__init__() 

        self.rnn = nn.RNN(inp_size, hid_size)
        self.h2o = nn.Linear(hid_size, outp_size) 
        self.softmax = nn.LogSoftmax(dim=1)

        self.gru = nn.GRU(
            input_size=inp_size, 
            hidd_size=hid_size_gru, 
            num_layers=num_layers, 
            batch_first=True, 
            dropuout=0.2
        )

        self.lstm = nn.LSTM( 
            input_size=hid_size_gru, 
            hidd_size = hid_size_lstm, 
            num_layers= 1, 
            batch_first=True
        )

        self.fc = nn.Linear(hid_size_lstm, outp_size) 

    def forward(self, line_tensr): 
        rnn_out, hidden  = self.rnn(line_tensr)
        outp = self.h2o(hidden[0]) 
        outp = self.softmax(outp)

        gru_out = self.gru(line_tensr) 

        lstm_out, (hn, cn) = self.lstm(gru_out) 
        outp = self.fc(lstm_out[:,-1, :])
        return outp

n_hidden = 256 
rnn = CharRNN(n_lets, n_hidden, len(alldata.labels_uniq))
print(rnn) 




In [ ]:
def label_from_output(output, output_labels): 
    top_n, top_i = output.topk(1) 
    label_i = top_i[0].item() 
    return output_labels[label_i], label_i

input = lineToTensor('Albert') 
output = rnn(input)
print(output) 
print(label_from_output, alldata.labels_uniq)

In [ ]:
import random 
import numpy as np 


def train(rnn,  training_data, n_epoch=10, n_batch_size=64, report=50, learn_rate=0.2, criterion=nn.NLLLoss()):
    current_loss = 0 
    all_losses = [] 
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learn_rate)

    start = time.time() 

    for iter in range(1, n_epoch + 1): 
        rnn.zero_grad() 

        batches = list(range(len(training_data))) 
        random.shuffle(batches) 
        batches = np.array_split(batches, len(batches) // n_batch_size)

        for idx, batch in enumerate(batches): 
            batch_loss = 0 

            for i in batch: 
                (label_tensor, txt_tensor, label, txt) = training_data[i]
                output = rnn.forward(txt_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss

            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step() 
            optimizer.zero_grad() 

            current_loss += batch_loss.item() / len(batch)
        all_losses.append(current_loss / len(batches)) 
        if iter % report == 0:
            print(f"{iter} ({iter / n_epoch:.0%}): \t average batch loss = {all_losses[-1]}")
        current_loss = 0 

    return all_losses

In [ ]:
start = time.time() 
all_losses = train(rnn, train_set, n_epoch=30, learn_rate=0.2, report=3)
end = time.time() 
print(f"training took {end-start}s")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.figure() 
plt.plot(all_losses) 
plt.show() 




In [ ]:

def evaluate(rnn, testing_data, classes): 
    confusion = torch.zeros(len(classes), len(classes))

    rnn.eval()

    with torch.no_grad(): 
        for t in range(len(testing_data)): 
            (label_tensor, txt_tensor, label, text) = testing_data[t] 
            output = rnn(txt_tensor) 
            guess, guess_i = label_from_output(output, classes) 
            label_i = classes.index(label) 
            confusion[label_i][guess_i] += 1

    for i in range(len(classes)):
        denom = confusion[i].sum() 
        if denom > 0: 
            confusion[i] = confusion[i] / denom 

    fig = plt.figure() 
    ax = fig.add_subplot(111) 
    cax = ax.matshow(confusion.cpu().numpy())
    fig.colorbar(cax) 

    ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90) 
    ax.set_yticks(np.arange(len(classes)), labels=classes)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1)) 
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show() 

evaluate(rnn, test_set, classes=alldata.labels_uniq) 

